# Dispersion Curve Validation Demo

This notebook mirrors `tests/test_core/test_dispersion.py` and adds plots to verify that `compute_dispersion_curve` is implemented correctly.

**Focus:** adaptive time-window length per frequency band and 50% overlap between consecutive windows.

Synthetic Love-wave data are used so the expected phase velocity (~3000 m/s) and backazimuth (45°) are known.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from obspy import Stream, Trace, UTCDateTime
from obspy.signal.rotate import rotate_ne_rt

from sixdegrees.sixdegrees import sixdegrees
from sixdegrees.plots.plot_dispersion_curves import plot_dispersion_curves
from sixdegrees.plots.plot_dispersion_traces import plot_dispersion_traces

plt.rcParams.update({"figure.figsize": (12, 4), "font.size": 11})


## Parameters (same as unit tests)


In [ ]:
# Signal / station setup
SR = 20.0                 # Hz
DURATION = 300.0            # s
SIGNAL_FREQ = 0.2         # Hz dominant signal
TRUE_BAZ = 45.0           # degrees
TRUE_VELOCITY = 3000.0    # m/s (Love-wave target)

# Dispersion processing
FMIN, FMAX = 0.1, 0.5     # Hz
OCTAVE_FRACTION = 3
WINDOW_FACTOR = 1.0       # T = max(window_factor / f_center, 1 s)
TIME_WINDOW_OVERLAP = 0.5 # 50% overlap
CC_THRESHOLD = 0.0
BAZ_STEP = 15


## Helper functions for window geometry


In [ ]:
def expected_window_count(n_samples, sampling_rate, time_window_sec, overlap):
    win_samples = int(time_window_sec * sampling_rate)
    overlap_samples = int(win_samples * overlap)
    step = win_samples - overlap_samples
    return int((n_samples - win_samples) / step) + 1


def expected_window_times(n_samples, sampling_rate, time_window_sec, overlap):
    win_samples = int(time_window_sec * sampling_rate)
    overlap_samples = int(win_samples * overlap)
    step = win_samples - overlap_samples
    n_windows = expected_window_count(n_samples, sampling_rate, time_window_sec, overlap)
    return np.array([(i * step + win_samples / 2) / sampling_rate for i in range(n_windows)])


def window_intervals(n_samples, sampling_rate, time_window_sec, overlap):
    """Return list of (t_start, t_end, t_center) for each sliding window."""
    win_samples = int(time_window_sec * sampling_rate)
    overlap_samples = int(win_samples * overlap)
    step = win_samples - overlap_samples
    n_windows = expected_window_count(n_samples, sampling_rate, time_window_sec, overlap)
    out = []
    for i in range(n_windows):
        i1 = i * step
        i2 = i1 + win_samples
        out.append((i1 / sampling_rate, i2 / sampling_rate, (i1 + win_samples / 2) / sampling_rate))
    return out


## 1. Build synthetic 6-DoF streams and `sixdegrees` object


In [ ]:
npts = int(DURATION * SR)
t = np.arange(npts) / SR

acc_n = np.sin(2 * np.pi * SIGNAL_FREQ * t)
acc_e = np.sin(2 * np.pi * SIGNAL_FREQ * t + 0.3)
_, acc_t = rotate_ne_rt(acc_n, acc_e, TRUE_BAZ)
rot_z = acc_t / (2 * TRUE_VELOCITY)  # slope ≈ TRUE_VELOCITY in regression

st_rot = Stream()
for ch, data in [
    ("BJZ", rot_z),
    ("BJN", np.sin(2 * np.pi * SIGNAL_FREQ * t + 0.1)),
    ("BJE", np.sin(2 * np.pi * SIGNAL_FREQ * t + 0.2)),
]:
    tr = Trace(data=data.astype(np.float64))
    tr.stats.sampling_rate = SR
    tr.stats.starttime = UTCDateTime(2024, 1, 1)
    tr.stats.network = "XX"
    tr.stats.station = "TEST"
    tr.stats.location = ""
    tr.stats.channel = ch
    st_rot += tr

st_acc = Stream()
for ch, data in [("BHZ", acc_n), ("BHN", acc_n), ("BHE", acc_e)]:
    tr = Trace(data=data.astype(np.float64))
    tr.stats.sampling_rate = SR
    tr.stats.starttime = UTCDateTime(2024, 1, 1)
    tr.stats.network = "XX"
    tr.stats.station = "TEST"
    tr.stats.location = ""
    tr.stats.channel = ch
    st_acc += tr

sd = sixdegrees({
    "tbeg": "2024-01-01",
    "tend": "2024-01-01T00:05:00",
    "fmin": FMIN,
    "fmax": FMAX,
    "fdsn_client_rot": "IRIS",
    "fdsn_client_tra": "IRIS",
    "rot_seed": "XX.TEST..BJZ",
    "tra_seed": "XX.TEST..BHZ",
})
sd.st0 = st_rot + st_acc
sd.st = sd.st0.copy()
sd.sampling_rate = SR
sd.baz_theo = TRUE_BAZ

print(f"Synthetic data: {DURATION:.0f} s @ {SR:.0f} Hz, true Love velocity = {TRUE_VELOCITY:.0f} m/s, BAz = {TRUE_BAZ:.0f}°")


## 2. Run `compute_dispersion_curve`


In [ ]:
results = sd.compute_dispersion_curve(
    wave_type="love",
    fmin=FMIN,
    fmax=FMAX,
    octave_fraction=OCTAVE_FRACTION,
    window_factor=WINDOW_FACTOR,
    time_window_overlap=TIME_WINDOW_OVERLAP,
    use_theoretical_baz=False,
    cc_threshold=CC_THRESHOLD,
    cc_method="max",
    baz_step=BAZ_STEP,
    verbose=True,
    n_jobs=1,
)

bands = results["frequency_bands"]
print(f"Computed {len(bands)} frequency bands")


## 3. Tabular check: adaptive window length and array sizes


In [ ]:
rows = []
for i, band in enumerate(bands):
    n_samples = len(band["filtered_rot"][0].data)
    tw = band["time_window"]
    expected_tw = max(WINDOW_FACTOR / band["f_center"], 1.0)
    expected_n = expected_window_count(n_samples, SR, tw, TIME_WINDOW_OVERLAP)
    rows.append({
        "band": i + 1,
        "f_center_Hz": band["f_center"],
        "f_range_Hz": f"{band['f_lower']:.3f}-{band['f_upper']:.3f}",
        "T_window_s": tw,
        "T_expected_s": expected_tw,
        "T_ok": np.isclose(tw, expected_tw),
        "n_windows": len(band["times"]),
        "n_expected": expected_n,
        "n_ok": len(band["times"]) == expected_n,
        "v_kde_m_s": band["kde_peak_velocity"],
        "baz_median_deg": np.nanmedian(band["backazimuths"]),
    })

summary = pd.DataFrame(rows)
display(summary.round({"f_center_Hz": 3, "T_window_s": 2, "T_expected_s": 2, "v_kde_m_s": 0, "baz_median_deg": 1}))

print("All T_window checks:", summary["T_ok"].all())
print("All window-count checks:", summary["n_ok"].all())


## 4. Adaptive window length vs center frequency

For `window_factor = 1`, the window length is `T = max(1/f_center, 1 s)` — longer windows at low frequency, shorter at high frequency.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
fc = summary["f_center_Hz"].values
T = summary["T_window_s"].values
T_theory = np.maximum(WINDOW_FACTOR / fc, 1.0)

ax.plot(fc, T, "o-", color="tab:blue", label="computed T_window")
ax.plot(fc, T_theory, "s--", color="tab:orange", label="max(window_factor/f_c, 1 s)")
ax.set_xlabel("Center frequency (Hz)")
ax.set_ylabel("Time window length (s)")
ax.set_title("Adaptive time-window length per band")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


## 5. Time windows with overlap (key validation plot)

For each selected band we draw:
- **coloured spans** = analysis windows (note 50% overlap)
- **black ticks** = window centres stored in `band['times']`
- **red dashed line** = filtered rotation trace (Love Z)

Zoom panels show the overlap clearly.


In [ ]:
def plot_band_windows(band, band_idx, tmax=None, ax=None):
    rot = band["filtered_rot"].select(channel="*Z")[0]
    times_trace = rot.times()
    data = rot.data

    tw = band["time_window"]
    n_samples = len(data)
    intervals = window_intervals(n_samples, SR, tw, TIME_WINDOW_OVERLAP)
    expected_centers = expected_window_times(n_samples, SR, tw, TIME_WINDOW_OVERLAP)

    if ax is None:
        _, ax = plt.subplots(figsize=(12, 3))

    cmap = plt.cm.imola
    for j, (t0, t1, tc) in enumerate(intervals):
        ax.axvspan(t0, t1, color=cmap(j / max(len(intervals) - 1, 1)), alpha=0.25)

    ax.plot(times_trace, data, color="0.2", lw=0.8, label="filtered rot-Z")
    ax.scatter(band["times"], np.zeros_like(band["times"]), marker="|", s=120, c="k", label="window centres (results)")
    ax.scatter(expected_centers, np.full_like(expected_centers, data.max() * 0.05), marker="v", s=25, c="red", label="expected centres")

    if tmax is not None:
        ax.set_xlim(0, tmax)

    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Amplitude")
    ax.set_title(
        f"Band {band_idx+1}: {band['f_lower']:.3f}-{band['f_upper']:.3f} Hz | "
        f"T={tw:.2f} s, overlap={TIME_WINDOW_OVERLAP:.0%}, n_windows={len(intervals)}"
    )
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(True, alpha=0.3)
    return ax

# Full record: lowest, middle, highest frequency bands
pick = [0, len(bands)//2, len(bands)-1]
fig, axes = plt.subplots(len(pick), 1, figsize=(12, 3*len(pick)), sharex=False)
if len(pick) == 1:
    axes = [axes]
for ax, bi in zip(axes, pick):
    plot_band_windows(bands[bi], bi, ax=ax)
plt.tight_layout()
plt.show()


In [ ]:
# Zoom first ~80 s on lowest-frequency band (longest windows, overlap easiest to see)
fig, ax = plt.subplots(figsize=(12, 3.5))
plot_band_windows(bands[0], 0, tmax=80, ax=ax)
ax.set_title(ax.get_title() + " — zoomed")
plt.tight_layout()
plt.show()

win_samples = int(bands[0]["time_window"] * SR)
step = win_samples - int(win_samples * TIME_WINDOW_OVERLAP)
print(f"Lowest band: window = {bands[0]['time_window']:.2f} s ({win_samples} samples)")
print(f"Step between window starts = {step/SR:.2f} s  →  overlap = {TIME_WINDOW_OVERLAP:.0%}")


## 6. Window centres match theory exactly


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

for ax, bi in zip(axes, pick):
    band = bands[bi]
    n_samples = len(band["filtered_rot"][0].data)
    tw = band["time_window"]
    expected = expected_window_times(n_samples, SR, tw, TIME_WINDOW_OVERLAP)
    diff = band["times"] - expected

    ax.plot(expected, diff, "o-", ms=4)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xlabel("Expected window centre (s)")
    ax.set_ylabel("computed − expected (s)")
    ax.set_title(f"Band {bi+1}: f_c={band['f_center']:.3f} Hz")
    ax.grid(True, alpha=0.3)
    print(f"Band {bi+1}: max |Δt| = {np.max(np.abs(diff)):.2e} s")

plt.suptitle("Window-centre consistency check", y=1.02)
plt.tight_layout()
plt.show()


## 7. Per-window velocities and backazimuths along time


In [ ]:
fig, axes = plt.subplots(len(pick), 1, figsize=(12, 3*len(pick)), sharex=True)
if len(pick) == 1:
    axes = [axes]

for ax, bi in zip(axes, pick):
    band = bands[bi]
    tw = band["time_window"]
    intervals = window_intervals(len(band["filtered_rot"][0].data), SR, tw, TIME_WINDOW_OVERLAP)

    for j, (t0, t1, _) in enumerate(intervals):
        ax.axvspan(t0, t1, color="0.9", alpha=0.5 if j % 2 else 0.3)

    v = band["velocities"]
    t = band["times"]
    valid = ~np.isnan(v)
    ax.plot(t[valid], v[valid], "o-", color="tab:blue", label="velocity estimate")
    ax.axhline(TRUE_VELOCITY, color="tab:green", ls=":", lw=2, label=f"true V = {TRUE_VELOCITY:.0f} m/s")
    ax.set_ylabel("V (m/s)")
    ax.set_title(f"Band {bi+1}: f_c={band['f_center']:.3f} Hz, KDE peak={band['kde_peak_velocity']:.0f} m/s")
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Window centre time (s)")
plt.tight_layout()
plt.show()


## 8. Package plot functions


In [ ]:
fig_traces = plot_dispersion_traces(
    results,
    data_type="acceleration",
    unitscale="nano",
    baz_theoretical=TRUE_BAZ,
)
plt.show()


In [ ]:
fig_curves = plot_dispersion_curves(
    dispersion_results=results,
    show_errors=True,
    xlog=True,
)
ax = fig_curves.axes[0]
ax.axhline(TRUE_VELOCITY, color="tab:green", ls=":", lw=2, label=f"true V = {TRUE_VELOCITY:.0f} m/s")
ax.legend()
plt.show()

print("KDE peak velocities (m/s):", np.round(results["velocities"], 0))
print("True velocity (m/s):", TRUE_VELOCITY)


## 9. Summary

If the procedure is correct you should observe:

1. **Adaptive windows:** `T_window = max(window_factor / f_center, 1 s)` for every band.
2. **Overlap:** consecutive windows overlap by `time_window_overlap` (50% here); zoom plot makes this visible.
3. **Alignment:** black window-centre markers coincide with red expected markers; difference plot is ~0.
4. **Counts:** `len(times) == len(velocities) == len(backazimuths)` and matches the sliding-window formula.
5. **Physics:** per-window and KDE velocities cluster near the known 3000 m/s for this synthetic Love wave.
